# SFVP AI Clone — Notebook 01: Voice Cloning
**Tool:** F5-TTS (MIT license — commercial use OK)

This notebook takes a short voice reference clip of Shennel and generates new audio in her voice from any script.

**Before running:** Make sure your Google Drive has this folder structure:
```
My Drive/SFVP_Clone/
├── voice_samples/   ← your .wav/.mp3/.m4a recordings here
└── outputs/         ← generated audio saves here
```

**Runtime:** Runtime → Change runtime type → T4 GPU

In [ ]:
# CELL 1 — Install F5-TTS (run once per Colab session, takes ~2 min)
!pip install git+https://github.com/SWivid/F5-TTS.git -q
!pip install soundfile -q
print('F5-TTS installed.')

In [ ]:
# CELL 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/SFVP_Clone'
os.makedirs(f'{DRIVE_BASE}/outputs', exist_ok=True)

# List available voice samples
samples_dir = f'{DRIVE_BASE}/voice_samples'
if os.path.exists(samples_dir):
    files = [f for f in os.listdir(samples_dir) if f.endswith(('.wav','.mp3','.m4a','.aac','.ogg'))]
    print(f'Found {len(files)} voice sample(s):')
    for i, f in enumerate(files):
        print(f'  [{i}] {f}')
else:
    print('ERROR: No voice_samples folder found in SFVP_Clone. Upload your recordings first.')

In [ ]:
# CELL 3 — CONFIGURE THIS CELL before running

# Pick which voice sample to use as the reference (use filename from list above)
REFERENCE_AUDIO_FILENAME = 'shennel_intro.wav'  # <-- CHANGE THIS

# The script text you want generated in Shennel's voice
SCRIPT = """
Real talk — if your business isn't turning heads, your branding is failing you.
Sactown's Finest does custom vinyl, banners, and print that actually stops the scroll.
Sacramento, link in bio. Let's get you seen.
"""  # <-- REPLACE WITH YOUR SCRIPT

# Output filename (saved to Drive/SFVP_Clone/outputs/)
OUTPUT_FILENAME = 'voice_output_01.wav'  # <-- CHANGE IF NEEDED

# Reference text (what's actually spoken in the reference clip)
# Leave empty ('') to auto-transcribe — works great, saves you typing it
REFERENCE_TEXT = ''  # <-- leave '' for auto-transcription

print('Config set.')
print(f'Reference audio: {REFERENCE_AUDIO_FILENAME}')
print(f'Output: {OUTPUT_FILENAME}')
print(f'Script ({len(SCRIPT.split())} words):\n{SCRIPT.strip()}')

In [ ]:
# CELL 4 — Convert reference audio to WAV if needed
import subprocess

ref_path = f'{DRIVE_BASE}/voice_samples/{REFERENCE_AUDIO_FILENAME}'
ref_wav = '/content/reference.wav'

if not os.path.exists(ref_path):
    print(f'ERROR: File not found: {ref_path}')
    print('Check the filename in Cell 3 matches exactly what you uploaded.')
else:
    # Convert to 24kHz WAV (F5-TTS works best at 24kHz)
    subprocess.run(['ffmpeg', '-y', '-i', ref_path, '-ar', '24000', '-ac', '1', ref_wav], 
                   capture_output=True)
    print(f'Reference audio ready: {ref_wav}')
    
    # Extract just the first 10 seconds for reference (F5-TTS uses short clips)
    ref_clip = '/content/reference_clip.wav'
    subprocess.run(['ffmpeg', '-y', '-i', ref_wav, '-t', '10', '-ar', '24000', '-ac', '1', ref_clip],
                   capture_output=True)
    print('10-second reference clip extracted.')

In [ ]:
# CELL 5 — Generate voice audio (this is the main step)
from f5_tts.api import F5TTS
import soundfile as sf

output_local = f'/content/{OUTPUT_FILENAME}'
output_drive = f'{DRIVE_BASE}/outputs/{OUTPUT_FILENAME}'

print('Loading F5-TTS model (first run downloads ~3GB, cached after)...')
tts = F5TTS()

print('Generating audio in Shennel\'s voice...')
wav, sr, _ = tts.infer(
    ref_file=ref_clip,
    ref_text=REFERENCE_TEXT,  # '' = auto-transcribe
    gen_text=SCRIPT.strip(),
    file_wave=output_local,
    seed=42,
    remove_silence=True,
)

# Copy to Drive
import shutil
shutil.copy(output_local, output_drive)

print(f'Done! Audio saved to:')
print(f'  Local (preview):  {output_local}')
print(f'  Google Drive:     {output_drive}')

In [ ]:
# CELL 6 — Preview the audio in Colab
from IPython.display import Audio, display
import soundfile as sf

print('GENERATED AUDIO — Listen and rate it:')
display(Audio(output_local))

print()
print('To improve quality:')
print('  1. Try a different voice sample as reference (run from Cell 3)')
print('  2. Use a cleaner, quieter recording as reference')
print('  3. Run Notebook 05 to fine-tune F5-TTS on all your samples (once you have 5+ min of audio)')

In [ ]:
# CELL 7 — (OPTIONAL) Try multiple reference clips and compare
# Run this to generate the same script with different reference clips side by side

samples_dir = f'{DRIVE_BASE}/voice_samples'
files = [f for f in os.listdir(samples_dir) if f.endswith(('.wav','.mp3','.m4a'))]

for i, sample_file in enumerate(files[:3]):  # compares first 3 samples
    ref = f'{samples_dir}/{sample_file}'
    ref_wav_i = f'/content/ref_{i}.wav'
    out_i = f'/content/compare_{i}.wav'
    
    subprocess.run(['ffmpeg', '-y', '-i', ref, '-t', '10', '-ar', '24000', '-ac', '1', ref_wav_i],
                   capture_output=True)
    
    tts.infer(ref_file=ref_wav_i, ref_text='', gen_text=SCRIPT.strip(),
              file_wave=out_i, seed=42, remove_silence=True)
    
    print(f'--- Sample [{i}]: {sample_file} ---')
    display(Audio(out_i))
    print()